# 🎙️ F5-TTS VIETNAMESE: 1-CLICK CLOUD GPU SERVER & VOICE CLONING (0 ĐỒNG)
Notebook này biến Google Colab T4 miễn phí thành một **Cloud GPU Server tốc độ cao** cho hệ thống Remotion.

### ⚡ QUY TRÌNH 1-CLICK TỰ ĐỘNG HÓA:
1. **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** -> **Save**.
2. **Chạy Bước 1:** Cài đặt môi trường & Vá lỗi tương thích `vinorm`.
3. **Chạy Bước 2:** Nạp Model ViVoice vào GPU (Model ~1.2 GB, tải 1 lần lưu vĩnh viễn trên Drive).
4. **Chạy Bước 3:** Khởi chạy **Cloud GPU Server** -> Nhận đường link công khai `https://xxxx.gradio.live`.
5. Tại máy tính của bạn, gõ 1 lệnh terminal duy nhất để sinh toàn bộ video:
   ```bash
   node tools/vox-tts.mjs <project> --provider f5 --url https://xxxx.gradio.live --voice mc_vantung
   ```

In [ ]:
# ==============================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE & THIẾT LẬP MÔI TRƯỜNG
# ==============================================================================
import os, glob
from google.colab import drive

print("🔗 Đang kết nối với Google Drive...")
drive.mount('/content/drive')

WORKSPACE_DIR = "/content/drive/MyDrive/F5_TTS_Workspace"
os.makedirs(f"{WORKSPACE_DIR}/models", exist_ok=True)
os.makedirs(f"{WORKSPACE_DIR}/voices", exist_ok=True)

# 1. Clone mã nguồn F5-TTS tối ưu cho tiếng Việt nếu chưa có
if not os.path.exists("/content/F5-TTS-Vietnamese-Google-Colab"):
    print("📥 Đang clone repository F5-TTS Vietnamese...")
    !git clone https://github.com/2tocom/F5-TTS-Vietnamese-Google-Colab.git /content/F5-TTS-Vietnamese-Google-Colab
    %cd /content/F5-TTS-Vietnamese-Google-Colab
    !pip install -q -r requirements.txt
    !pip install -q huggingface_hub soundfile torchaudio fastapi uvicorn
else:
    %cd /content/F5-TTS-Vietnamese-Google-Colab

# 2. Vá lỗi vinorm cho Python 3.12/3.13 trên Google Colab
for init_file in glob.glob('/usr/local/lib/python3*/dist-packages/vinorm/__init__.py'):
    with open(init_file, 'r', encoding='utf-8') as f:
        content = f.read()
    content = content.replace("import imp", "").replace("A=imp.find_module('vinorm')[1]", "A=os.path.dirname(__file__)")
    with open(init_file, 'w', encoding='utf-8') as f:
        f.write(content)
print("✅ Hoàn tất thiết lập môi trường và vá lỗi vinorm!")

### 🧠 BƯỚC 2: NẠP MODEL TIẾNG VIỆT VIVOICE
Model được lưu trữ trên Google Drive để không phải tải lại mỗi lần mở Colab.

In [ ]:
# ==============================================================================
# BƯỚC 2: NẠP MODEL VIVOICE VÀ VOCODER
# ==============================================================================
import os
import torch
from huggingface_hub import snapshot_download
from f5_tts.model import DiT
from f5_tts.infer.utils_infer import (
    preprocess_ref_audio_text,
    load_vocoder,
    load_model,
    infer_process,
    chunk_text,
    process_text_chunks,
)

DRIVE_MODEL_DIR = f"{WORKSPACE_DIR}/models/F5-TTS-Vietnamese-ViVoice"
ckpt_file = os.path.join(DRIVE_MODEL_DIR, "model_last.pt")
vocab_file = os.path.join(DRIVE_MODEL_DIR, "config.json")

# Tải model vào Google Drive nếu chưa tồn tại
if not os.path.exists(ckpt_file):
    print("⏳ Chưa có model trên Google Drive. Đang tải từ Hugging Face (1.2 GB, chỉ tải 1 lần)... ")
    snapshot_download(repo_id="hynt/F5-TTS-Vietnamese-ViVoice", local_dir=DRIVE_MODEL_DIR)
    print("✅ Đã tải và lưu model vĩnh viễn vào Google Drive!")
else:
    print("⚡ Đã tìm thấy model trên Google Drive! Nạp trực tiếp...")

print("🚀 Đang nạp Model vào GPU...")
vocoder = load_vocoder()
model = load_model(
    DiT,
    dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4),
    ckpt_path=ckpt_file,
    vocab_file=vocab_file,
)
print("✅ Model và Vocoder đã sẵn sàng tạo giọng!")

### 🚀 BƯỚC 3: KHỞI CHẠY CLOUD GPU SERVER (GIẢI PHÁP 1: API TỰ ĐỘNG TỪ MÁY TÍNH)
Chạy ô này để khởi động Server. Sau khi chạy, bạn sẽ nhận được một đường link dạng `https://xxxxxxxxxxxx.gradio.live`.
Chỉ cần copy đường link đó và chạy lệnh trên máy tính của bạn, mọi câu thoại sẽ tự động sinh và nạp thẳng vào dự án!

In [ ]:
# ==============================================================================
# BƯỚC 3: KHỞI CHẠY F5-TTS CLOUD GPU SERVER (REST API + GRADIO WEB UI)
# ==============================================================================
import os, sys, json, base64, io, re
import soundfile as sf
from fastapi import Request
from fastapi.responses import JSONResponse
import gradio as gr

VOICES_DIR = "/content/voices"
os.makedirs(VOICES_DIR, exist_ok=True)
VOICE_CACHE = {}

def load_voice(name):
    if name in VOICE_CACHE:
        return VOICE_CACHE[name]
    v_dir = os.path.join(VOICES_DIR, name)
    wav_path = os.path.join(v_dir, "reference.wav")
    txt_path = os.path.join(v_dir, "reference.txt")
    if not os.path.exists(wav_path):
        # Kiểm tra thêm trong Google Drive nếu có
        drive_v = f"{WORKSPACE_DIR}/voices/{name}"
        if os.path.exists(os.path.join(drive_v, "reference.wav")):
            wav_path = os.path.join(drive_v, "reference.wav")
            txt_path = os.path.join(drive_v, "reference.txt")
        else:
            return None
    ref_text = ""
    if os.path.exists(txt_path):
        with open(txt_path, "r", encoding="utf-8") as f:
            ref_text = f.read().strip()
    ref_audio, ref_text_final = preprocess_ref_audio_text(wav_path, ref_text)
    VOICE_CACHE[name] = (ref_audio, ref_text_final)
    return VOICE_CACHE[name]

def post_process_text(text):
    silence_pattern = r'\s*<<<\s*sil#(\d{1,5})\s*>>>\s*'
    silence_tokens = []
    def replace_silence(match):
        ms = int(match.group(1))
        ms = max(100, min(20000, int(round(ms / 100.0) * 100)))
        placeholder = f"__SILENCE_TOKEN_{len(silence_tokens)}__"
        silence_tokens.append(f" <<<sil#{ms}>>> ")
        return placeholder
    text_processed = re.sub(silence_pattern, replace_silence, text)
    text_processed = " " + text_processed + " "
    text_processed = text_processed.replace('"', "")
    text_processed = " ".join(text_processed.split())
    for i, token in enumerate(silence_tokens):
        text_processed = text_processed.replace(f"__SILENCE_TOKEN_{i}__", token)
    return text_processed

def synthesize_core(text, ref_audio, ref_text_final, speed=1.0):
    processed = post_process_text(text)
    chunks = chunk_text(processed)
    final_chunks = process_text_chunks(chunks)
    final_wave, final_sample_rate, _ = infer_process(
        ref_audio, ref_text_final.lower(), final_chunks, model, vocoder, speed=speed
    )
    buf = io.BytesIO()
    sf.write(buf, final_wave, final_sample_rate, format='WAV')
    buf.seek(0)
    duration = round(len(final_wave) / float(final_sample_rate), 3)
    return buf.read(), duration, final_sample_rate, final_wave

# 1. Giao diện Web UI Gradio (Dùng thử trên trình duyệt)
with gr.Blocks(title="F5-TTS Vietnamese Cloud GPU Server") as demo:
    gr.Markdown("# 🎙️ F5-TTS VIETNAMESE CLOUD GPU SERVER (API & WEB UI)")
    with gr.Row():
        ref_audio = gr.Audio(label="🔊 Sample Voice", type="filepath")
        gen_text = gr.Textbox(label="📝 Text", placeholder="Nhập văn bản cần đọc...", lines=3)
    ref_text = gr.Textbox(label="📝 Reference Text (optional)", placeholder="Nguyên văn câu trong file mẫu", lines=2)
    speed = gr.Slider(0.3, 2.0, value=1.0, step=0.1, label="⚡ Speed")
    btn = gr.Button("🔥 Generate Voice", variant="primary")
    with gr.Row():
        out_audio = gr.Audio(label="🎧 Generated Audio", type="numpy")
    
    def gradio_infer(audio_file, text, ref_txt, spd):
        if not audio_file or not text.strip():
            return None
        r_audio, r_txt = preprocess_ref_audio_text(audio_file, ref_txt)
        _, _, sr, wave = synthesize_core(text, r_audio, r_txt, spd)
        return (sr, wave)

    btn.click(gradio_infer, inputs=[ref_audio, gen_text, ref_text, speed], outputs=[out_audio])

# 2. Tích hợp High-Speed REST API vào FastAPI của Gradio
app = demo.app

@app.post("/api/voice")
async def register_voice(req: Request):
    """Đồng bộ và cache mẫu giọng từ máy local lên Colab"""
    body = await req.json()
    name = body.get("name", "custom")
    ref_text = body.get("ref_text", "")
    audio_b64 = body.get("audio_base64", "")
    if not audio_b64:
        return JSONResponse({"error": "Thiếu audio_base64"}, status_code=400)
    
    v_dir = os.path.join(VOICES_DIR, name)
    os.makedirs(v_dir, exist_ok=True)
    wav_path = os.path.join(v_dir, "reference.wav")
    with open(wav_path, "wb") as f:
        f.write(base64.b64decode(audio_b64))
    if ref_text:
        with open(os.path.join(v_dir, "reference.txt"), "w", encoding="utf-8") as f:
            f.write(ref_text)
    
    ref_audio, ref_text_final = preprocess_ref_audio_text(wav_path, ref_text)
    VOICE_CACHE[name] = (ref_audio, ref_text_final)
    print(f"✅ Đã nạp và kích hoạt giọng: {name}")
    return {"status": "ok", "voice": name}

@app.post("/api/tts")
async def api_tts(req: Request):
    """Tạo âm thanh từ xa cho Remotion Client"""
    body = await req.json()
    text = body.get("text", "").strip()
    voice = body.get("voice", "mc_vantung")
    speed = float(body.get("speed", 1.0))
    if not text:
        return JSONResponse({"error": "Thiếu text"}, status_code=400)
    
    v_info = load_voice(voice)
    if not v_info:
        return JSONResponse({"error": f"Chưa tìm thấy giọng '{voice}' trên server"}, status_code=404)
    
    ref_audio, ref_text_final = v_info
    audio_bytes, duration, _, _ = synthesize_core(text, ref_audio, ref_text_final, speed=speed)
    
    return {
        "status": "ok",
        "duration": duration,
        "audio_base64": base64.b64encode(audio_bytes).decode("utf-8")
    }

# 3. Khởi chạy Server công khai
print("\n" + "="*70)
print("🚀 F5-TTS CLOUD GPU SERVER ĐANG KHỞI ĐỘNG...")
print("="*70)
demo.queue().launch(share=True, show_error=True)